# Small case demo: LLM routing and deployment optimization

This deterministic, provider-neutral notebook exercises the complete `llm-route-opt` MVP using the checked-in synthetic RouterBench data. Run it from the repository root after `python -m pip install -e .`. No API key or network access is needed.

In [ ]:
from dataclasses import asdict
from pathlib import Path

from llm_route_opt import (
    CascadeRouter,
    DAGRouter,
    DeploymentOptimizer,
    SingleModelRouter,
    TopKRouter,
    Workload,
    evaluate,
    maximize_quality,
)
from llm_route_opt.benchmark import load_routerbench_jsonl
from llm_route_opt.inverse import inverse_example
from llm_route_opt.queueing import fcfs_mm_c
from llm_route_opt.routers import DAGEdge

data_path = Path("examples/synthetic_routerbench.jsonl")
if not data_path.exists():  # Also works when Jupyter starts in examples/.
    data_path = Path("synthetic_routerbench.jsonl")
dataset = load_routerbench_jsonl(data_path)
models = tuple(dataset.models.values())
print(
    f"Loaded {len(dataset.queries)} queries, {len(models)} models, and {len(dataset.measurements)} measurements."
)

## 1. Compare single-model, top-k, cascade, and DAG routers

The top-k policy executes its first ranked candidate. The cascade escalates until its predicted quality reaches 0.80. The DAG uses deterministic, ordered difficulty predicates.

In [ ]:
dag = DAGRouter(
    dataset.models,
    root="small",
    edges=[
        DAGEdge("small", "medium", lambda query: query.difficulty >= 0.40, "medium difficulty"),
        DAGEdge("medium", "large", lambda query: query.difficulty >= 0.80, "hard query"),
    ],
)
routers = {
    "single-medium": SingleModelRouter(dataset.models["medium"]),
    "top-2": TopKRouter(models, k=2),
    "cascade": CascadeRouter(models, quality_threshold=0.80),
    "dag": dag,
}

results = {name: evaluate(dataset, router) for name, router in routers.items()}
print(f"{'router':<16} {'quality':>9} {'cost ($)':>10} {'latency':>12} {'models'}")
for name, result in results.items():
    print(
        f"{name:<16} {result.mean_quality:>9.4f} {result.total_cost:>10.5f} "
        f"{result.mean_latency_ms:>9.1f} ms  {result.model_counts}"
    )

Inspecting route decisions makes cascade and graph behavior explicit.

In [ ]:
for query in dataset.queries:
    cascade_decision = routers["cascade"].route(query)
    dag_decision = routers["dag"].route(query)
    print(
        query.query_id,
        f"difficulty={query.difficulty:.2f}",
        f"cascade={cascade_decision.path}",
        f"dag={dag_decision.path}",
    )

## 2. Maximize quality under cost and latency constraints

This small finite case is solved exactly. Every query receives one model, total predicted cost is at most $0.01, and models slower than 600 ms are excluded.

In [ ]:
routing_plan = maximize_quality(
    dataset.queries,
    models,
    total_budget=0.01,
    max_latency_ms=600.0,
)
print(asdict(routing_plan))

## 3. FCFS waiting time and deployment assignment

First inspect one M/M/c queue, then jointly assign two workload classes and choose stable integer replica counts.

In [ ]:
queue = fcfs_mm_c(
    arrival_rate_rps=5.0,
    service_rate_rps=4.0,
    servers=2,
    service_latency_ms=210.0,
)
print("FCFS queue:", asdict(queue))

deployment_optimizer = DeploymentOptimizer(
    models,
    replica_hourly_cost={"small": 0.40, "medium": 1.10, "large": 3.50},
    max_replicas_per_model=8,
)
deployment_plan = deployment_optimizer.optimize(
    workloads=[
        Workload("interactive", arrival_rate_rps=5.0, min_quality=0.70, max_response_ms=500.0),
        Workload("reasoning", arrival_rate_rps=0.8, min_quality=0.90, max_response_ms=1500.0),
    ],
    hourly_budget=8.0,
)
print("Deployment:", asdict(deployment_plan))

## 4. Infer objective weights from discrete choices

The inverse example finds non-negative quality, economy, and speed weights that sum to one and rationalize all observed choices.

In [ ]:
inverse_solution = inverse_example()
print(asdict(inverse_solution))
assert inverse_solution.pairwise_accuracy == 1.0
assert abs(sum(inverse_solution.weights.values()) - 1.0) < 1e-12

## Next steps

Replace the synthetic JSONL file with normalized per-query/per-model measurements, or inject a learned quality estimator into the routers and optimizer. Keep estimator training queries separate from evaluation queries to avoid leakage.